# Prompt Template Sensitivity Study
## VLM Medical VQA Benchmark

**Purpose:** Evaluate whether the `v2` prompt template (which includes radiology-specific framing and 'Final Answer: X' constraints) systematically underestimates models like HuatuoGPT-7B (Qwen2.5-VL backbone) or LLaVA-Med-7B (conversational PMC-15M training). We test 3 prompt variants on a 200-sample subset of VQA-RAD.

**Models to test:**
- `FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL`
- `microsoft/llava-med-v1.5-mistral-7b`

**Setup for Kaggle:**
Run this on a Kaggle T4 GPU (or P100) and download the generated `outputs/` folder. Repeat for both models by changing `MODEL_ID`.

In [ ]:
!pip install -q transformers==4.51.3 datasets accelerate bitsandbytes sentencepiece pillow flash_attn
import os
os.makedirs('outputs', exist_ok=True)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
import json
import re
from tqdm.auto import tqdm

# ==========================================
# 1. MODEL CONFIGURATION
# ==========================================
# Change this to 'microsoft/llava-med-v1.5-mistral-7b' for the second run
MODEL_ID = 'FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL'
# MODEL_ID = 'microsoft/llava-med-v1.5-mistral-7b'

model_name_safe = MODEL_ID.replace('/', '_')

In [ ]:
# ==========================================
# 2. LOAD MODEL
# ==========================================
print(f"Loading {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    trust_remote_code=True
)
print("Model loaded successfully!")

In [ ]:
# ==========================================
# 3. LOAD DATASET (200-sample subset)
# ==========================================
ds = load_dataset('flaviagiammarino/vqa-rad', split='test')
ds = ds.shuffle(seed=42).select(range(200))

def is_closed_question(answer):
    return str(answer).strip().lower() in ['yes', 'no']

samples = []
for i, item in enumerate(ds):
    samples.append({
        'idx': i,
        'image': item['image'],
        'question': item['question'],
        'answer': str(item['answer']),
        'is_closed': is_closed_question(item['answer'])
    })

print(f"Loaded {len(samples)} samples.")

In [ ]:
# ==========================================
# 4. PROMPT VARIANTS
# ==========================================

def build_v2_baseline(question: str, is_closed: bool) -> str:
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return (
        f"{prefix}Given this radiology image, which can be a frontal chest X-ray, "
        f"a single slice head or abdominal CT or MR image, provide a very short, "
        f"definitive, and concise answer (if possible, a single word) "
        f"to the following question: {question}"
    )

def build_v3_simple(question: str, is_closed: bool) -> str:
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return f"{prefix}Question: {question}\nAnswer the question directly and concisely."

def build_v4_chat(question: str, is_closed: bool) -> str:
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return f"{prefix}{question}\nProvide your answer as a single short phrase starting with 'Answer:'."

def extract_answer(text: str, variant_name: str) -> str:
    if variant_name == 'v4_chat':
        match = re.search(r'[Aa]nswer\s*:\s*(.+)', text, re.DOTALL)
        if match:
            ans = match.group(1).strip().split('\n')[0]
            return re.sub(r'[\*"\']+', '', ans).strip()
    
    # Fallback to first line
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return lines[0] if lines else text.strip()

VARIANTS = {
    'v2_baseline': build_v2_baseline,
    'v3_simple': build_v3_simple,
    'v4_chat': build_v4_chat
}

In [ ]:
# ==========================================
# 5. INFERENCE LOOP
# ==========================================
for variant_name, builder_fn in VARIANTS.items():
    print(f"\n--- Running Variant: {variant_name} ---")
    out_path = f"outputs/{model_name_safe}__{variant_name}.jsonl"
    
    with open(out_path, 'w') as f_out:
        for sample in tqdm(samples, desc=variant_name):
            prompt_text = builder_fn(sample['question'], sample['is_closed'])
            
            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': sample['image']},
                    {'type': 'text',  'text': prompt_text},
                ]
            }]
            
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=text, images=sample['image'], return_tensors='pt').to(model.device)
            
            with torch.inference_mode():
                output_ids = model.generate(**inputs, max_new_tokens=30, do_sample=False)
            
            input_len = inputs['input_ids'].shape[-1]
            raw_out = processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()
            prediction = extract_answer(raw_out, variant_name)
            
            record = {
                'idx': sample['idx'],
                'question': sample['question'],
                'ground_truth': sample['answer'],
                'prediction': prediction,
                'raw_output': raw_out,
                'is_closed': sample['is_closed'],
                'variant': variant_name
            }
            f_out.write(json.dumps(record) + '\n')
    
    print(f"Saved to {out_path}")